# LeetCode 521 — Longest Uncommon Subsequence I bằng Python

**Link đề:** https://leetcode.com/problems/longest-uncommon-subsequence-i/
**Độ khó:** Easy · **Chủ đề:** String

```python
class Solution:
    def findLUSlength(self, a: str, b: str) -> int:
        ...
```

## Mục tiêu học
1. Hiểu chính xác **subsequence** (dãy con — giữ thứ tự, được bỏ ký tự) khác **substring** (chuỗi con liên tiếp).
2. Viết hàm `is_subsequence(s, t)` bằng **hai con trỏ** (two pointers) và sinh mọi subsequence bằng `itertools.combinations`.
3. Làm brute force theo đúng định nghĩa (chỉ chạy được với chuỗi ngắn) để **thấy** đáp án có quy luật.
4. Rút ra insight "**bài mẹo**": `a == b` → `-1`, ngược lại `max(len(a), len(b))`, và **chứng minh** nó.
5. Đối chiếu brute force với công thức trên hàng nghìn input; luyện biến thể LUS II (nhiều chuỗi).

## 1. Đề bài

Cho hai chuỗi `a` và `b`, trả về **độ dài của uncommon subsequence dài nhất** giữa chúng. Nếu không tồn tại, trả `-1`.

**Uncommon subsequence** giữa `a` và `b` là một chuỗi là subsequence của **đúng một** trong hai chuỗi (của chuỗi này nhưng **không** của chuỗi kia).

| a | b | Output | Giải thích |
|---|---|---:|---|
| `"aba"` | `"cdc"` | 3 | `"aba"` là subsequence của `a` nhưng không của `b` |
| `"aaa"` | `"bbb"` | 3 | `"aaa"` và `"bbb"` đều là đáp án |
| `"aaa"` | `"aaa"` | -1 | mọi subsequence của chuỗi này cũng là của chuỗi kia |

**Constraints:** `1 <= a.length, b.length <= 100`; chỉ gồm chữ thường tiếng Anh.

**Hint của LeetCode:** "Think very simple." 🙂 — nhưng ta sẽ đi từng bước để **hiểu vì sao** đơn giản.

## 2. Nền tảng

### 2.1. Subsequence vs substring

- **Substring** (chuỗi con): các ký tự **liên tiếp**. `"bc"` là substring của `"abcd"`.
- **Subsequence** (dãy con): lấy một số ký tự, **giữ nguyên thứ tự**, **được bỏ** ký tự ở giữa. `"ad"` là subsequence của `"abcd"` (bỏ `b, c`) nhưng **không** là substring.
- Chuỗi rỗng `""` là subsequence của mọi chuỗi; mỗi chuỗi là subsequence của chính nó.

In [1]:
s = "abcd"
print("'bc' substring?", "bc" in s)     # True
print("'ad' substring?", "ad" in s)     # False — nhưng 'ad' LÀ subsequence

'bc' substring? True
'ad' substring? False


### 2.2. Kiểm tra subsequence bằng hai con trỏ

Ý tưởng: con trỏ `i` chạy trên `s` (chuỗi cần tìm), con trỏ `j` chạy trên `t`. Mỗi bước `j` tiến 1; nếu `t[j] == s[i]` thì `i` cũng tiến. Nếu `i` đi hết `s` → `s` là subsequence của `t`.

Tham lam (greedy) khớp **sớm nhất có thể** là an toàn: khớp sớm không bao giờ làm mất cơ hội cho các ký tự sau.

In [2]:
def is_subsequence(s, t):
    i = 0
    for ch in t:
        if i < len(s) and s[i] == ch:
            i += 1
    return i == len(s)

print(is_subsequence("ad", "abcd"))    # True
print(is_subsequence("da", "abcd"))    # False — sai thứ tự
print(is_subsequence("", "abc"))       # True
print(is_subsequence("aba", "cdc"))    # False

True
False
True
False


### 2.3. Sinh mọi subsequence — `itertools.combinations`

Một subsequence độ dài `k` ứng với việc **chọn `k` chỉ số tăng dần** trong chuỗi. `combinations(range(n), k)` sinh mọi bộ chỉ số như vậy. Chuỗi độ dài `n` có `2ⁿ` cách chọn (tính cả rỗng) → chỉ dùng được khi `n` nhỏ.

In [3]:
from itertools import combinations

def all_subsequences(s):
    result = set()
    for k in range(len(s) + 1):
        for idx in combinations(range(len(s)), k):
            result.add("".join(s[i] for i in idx))
    return result

print(sorted(all_subsequences("aba"), key=lambda x: (len(x), x)))
print(len(all_subsequences("abcd")), "= 2^4 (chuỗi không lặp ký tự)")

['', 'a', 'b', 'aa', 'ab', 'ba', 'aba']
16 = 2^4 (chuỗi không lặp ký tự)


## 3. Cách 1 — Brute force theo đúng định nghĩa

1. Sinh tập `SA` = mọi subsequence của `a`, `SB` = mọi subsequence của `b`.
2. Uncommon = chuỗi thuộc **đúng một** tập = **hiệu đối xứng** `SA ^ SB` (phép `^` trên set).
3. Trả độ dài lớn nhất, hoặc `-1` nếu rỗng.

In [4]:
def lus_bruteforce(a, b):
    SA, SB = all_subsequences(a), all_subsequences(b)
    uncommon = SA ^ SB                  # thuộc đúng một trong hai
    return max((len(x) for x in uncommon), default=-1)

print(lus_bruteforce("aba", "cdc"))   # 3
print(lus_bruteforce("aaa", "bbb"))   # 3
print(lus_bruteforce("aaa", "aaa"))   # -1

3
3
-1


`max(..., default=-1)` trả `-1` khi không có phần tử nào — tránh lỗi `ValueError` của `max` trên dãy rỗng.

Phép **hiệu đối xứng** `A ^ B` = `(A - B) | (B - A)`:

In [5]:
A, B = {1, 2, 3}, {3, 4}
print(A ^ B, (A - B) | (B - A))

{1, 2, 4} {1, 2, 4}


### Nút thắt

`a` dài 100 → `2^100` subsequence: **không thể** liệt kê. Nhưng brute force trên chuỗi ngắn giúp ta **quan sát quy luật**. Hãy chạy thử nhiều cặp:

In [6]:
pairs = [("aba", "cdc"), ("aaa", "bbb"), ("aaa", "aaa"), ("ab", "abc"), ("abc", "ab"),
         ("aefd", "ab"), ("ab", "ba"), ("a", "a"), ("abcd", "abce")]
for a, b in pairs:
    print(f"a={a!r:7} b={b!r:7} brute={lus_bruteforce(a, b):2d} | a==b? {a == b!s:5} | max len = {max(len(a), len(b))}")

a='aba'   b='cdc'   brute= 3 | a==b? False | max len = 3
a='aaa'   b='bbb'   brute= 3 | a==b? False | max len = 3
a='aaa'   b='aaa'   brute=-1 | a==b? True  | max len = 3
a='ab'    b='abc'   brute= 3 | a==b? False | max len = 3
a='abc'   b='ab'    brute= 3 | a==b? False | max len = 3
a='aefd'  b='ab'    brute= 4 | a==b? False | max len = 4
a='ab'    b='ba'    brute= 2 | a==b? False | max len = 2
a='a'     b='a'     brute=-1 | a==b? True  | max len = 1
a='abcd'  b='abce'  brute= 4 | a==b? False | max len = 4


Quy luật: **đáp án luôn là `-1` khi `a == b`, còn lại luôn là `max(len(a), len(b))`**. Bây giờ ta chứng minh điều này.

## 4. Insight — chuỗi dài hơn chính là đáp án

**Trường hợp `a == b`:** mọi subsequence của `a` cũng là subsequence của `b` và ngược lại → không có uncommon subsequence → `-1`.

**Trường hợp `a != b`:** giả sử `len(a) >= len(b)` (vai trò đối xứng).
- Xét chính chuỗi `a` (là subsequence của `a`). Nó có là subsequence của `b` không?
  - Nếu `len(a) > len(b)`: không thể — subsequence không dài hơn chuỗi gốc.
  - Nếu `len(a) == len(b)`: subsequence cùng độ dài với `b` thì phải **bằng** `b`; mà `a != b` → không.
- Vậy `a` là uncommon subsequence, độ dài `len(a)`.
- Không thể dài hơn: mọi subsequence của `a` hoặc `b` có độ dài `<= max(len(a), len(b)) = len(a)`.

⇒ Đáp án `= max(len(a), len(b))`.

In [7]:
a, b = "abcd", "abce"
print("a là subsequence của b?", is_subsequence(a, b))
print("b là subsequence của a?", is_subsequence(b, a))
print("=> cả a và b đều là uncommon, đáp án =", max(len(a), len(b)))

a là subsequence của b? False
b là subsequence của a? False
=> cả a và b đều là uncommon, đáp án = 4


## 5. Lời giải chính

In [8]:
def lus_formula(a, b):
    if a == b:
        return -1
    return max(len(a), len(b))

for a, b in pairs:
    print(f"{a!r:7} {b!r:7} -> {lus_formula(a, b)}")

'aba'   'cdc'   -> 3
'aaa'   'bbb'   -> 3
'aaa'   'aaa'   -> -1
'ab'    'abc'   -> 3
'abc'   'ab'    -> 3
'aefd'  'ab'    -> 4
'ab'    'ba'    -> 2
'a'     'a'     -> -1
'abcd'  'abce'  -> 4


### Cách 2b — Kiểm tra trực tiếp ứng viên dài nhất (tư duy dùng lại cho bài LUS II)

Không cần "đoán" công thức: ứng viên tốt nhất luôn là **chính một trong hai chuỗi**. Thử chuỗi dài trước, kiểm tra bằng `is_subsequence`:

In [9]:
def lus_check_candidates(a, b):
    best = -1
    for cand, other in [(a, b), (b, a)]:
        if not is_subsequence(cand, other):
            best = max(best, len(cand))
    return best

for a, b in pairs:
    print(f"{a!r:7} {b!r:7} -> {lus_check_candidates(a, b)}")

'aba'   'cdc'   -> 3
'aaa'   'bbb'   -> 3
'aaa'   'aaa'   -> -1
'ab'    'abc'   -> 3
'abc'   'ab'    -> 3
'aefd'  'ab'    -> 4
'ab'    'ba'    -> 2
'a'     'a'     -> -1
'abcd'  'abce'  -> 4


## 6. TRACE từng bước

Trace phiên bản kiểm tra ứng viên với `a = "ab"`, `b = "abc"` (a ngắn hơn và là subsequence của b):

| ứng viên | so với | hai con trỏ khớp được | là subsequence? | uncommon? | best |
|---|---|---|---|---|---:|
| `"ab"` | `"abc"` | `a`, `b` → i = 2 = len | có | không | -1 |
| `"abc"` | `"ab"` | `a`, `b` → i = 2 < 3 | không | **có** | 3 |

In [10]:
def trace_is_subsequence(s, t):
    i = 0
    for j, ch in enumerate(t):
        hit = i < len(s) and s[i] == ch
        print(f"      j={j} t[j]='{ch}' s[i]={s[i] if i < len(s) else '—'!r:5} khớp={hit}")
        if hit:
            i += 1
    print(f"      -> i={i}, len(s)={len(s)}, subsequence={i == len(s)}")
    return i == len(s)

def trace_lus(a, b):
    best = -1
    for cand, other in [(a, b), (b, a)]:
        print(f"ứng viên {cand!r} so với {other!r}:")
        if trace_is_subsequence(cand, other):
            print("   => không uncommon")
        else:
            best = max(best, len(cand))
            print(f"   => uncommon, best = {best}")
    print("KẾT QUẢ:", best)
    return best

trace_lus("ab", "abc")

ứng viên 'ab' so với 'abc':
      j=0 t[j]='a' s[i]='a'   khớp=True
      j=1 t[j]='b' s[i]='b'   khớp=True
      j=2 t[j]='c' s[i]='—'   khớp=False
      -> i=2, len(s)=2, subsequence=True
   => không uncommon
ứng viên 'abc' so với 'ab':
      j=0 t[j]='a' s[i]='a'   khớp=True
      j=1 t[j]='b' s[i]='b'   khớp=True
      -> i=2, len(s)=3, subsequence=False
   => uncommon, best = 3
KẾT QUẢ: 3


3

In [11]:
trace_lus("aaa", "aaa")

ứng viên 'aaa' so với 'aaa':
      j=0 t[j]='a' s[i]='a'   khớp=True
      j=1 t[j]='a' s[i]='a'   khớp=True
      j=2 t[j]='a' s[i]='a'   khớp=True
      -> i=3, len(s)=3, subsequence=True
   => không uncommon
ứng viên 'aaa' so với 'aaa':
      j=0 t[j]='a' s[i]='a'   khớp=True
      j=1 t[j]='a' s[i]='a'   khớp=True
      j=2 t[j]='a' s[i]='a'   khớp=True
      -> i=3, len(s)=3, subsequence=True
   => không uncommon
KẾT QUẢ: -1


-1

## 7. Code nộp LeetCode

In [12]:
class Solution:
    def findLUSlength(self, a: str, b: str) -> int:
        return -1 if a == b else max(len(a), len(b))

s = Solution()
print(s.findLUSlength("aba", "cdc"), s.findLUSlength("aaa", "bbb"), s.findLUSlength("aaa", "aaa"))

3 3 -1


## 8. Vì sao đúng? (tóm tắt)

- **Giới hạn trên:** subsequence của một chuỗi không dài hơn chuỗi đó → mọi uncommon subsequence có độ dài `≤ max(len(a), len(b))`.
- **Đạt được** khi `a != b`: chuỗi dài hơn (hoặc bất kỳ chuỗi nào nếu cùng độ dài) không phải subsequence của chuỗi kia → nó là uncommon với độ dài đúng bằng giới hạn trên.
- **`a == b`:** hai tập subsequence trùng nhau → hiệu đối xứng rỗng → `-1`.

## 9. Độ phức tạp

`n = len(a)`, `m = len(b)`.

| Cách | Time | Space | Ghi chú |
|---|---:|---:|---|
| Brute force sinh mọi subsequence | `O((2ⁿ + 2ᵐ)·(n + m))` | `O(2ⁿ + 2ᵐ)` | chỉ để quan sát, n ≤ ~15 |
| Kiểm tra 2 ứng viên | `O(n + m)` | `O(1)` | tư duy dùng lại cho LUS II |
| Công thức | `O(min(n, m))` cho phép so sánh `a == b` | `O(1)` | ⭐ |

(So sánh chuỗi `a == b` dừng ngay nếu độ dài khác nhau hoặc gặp ký tự khác.)

Để cảm nhận brute force bùng nổ thế nào:

In [13]:
import time
for n in [8, 12, 16]:
    s1 = "".join(chr(97 + i % 26) for i in range(n))
    t0 = time.perf_counter(); cnt = len(all_subsequences(s1)); t1 = time.perf_counter()
    print(f"n={n:2d}: {cnt:6d} subsequence, sinh mất {t1 - t0:.3f} s")
print("n=100 sẽ có tới 2^100 ≈", f"{2**100:.2e}", "subsequence — không thể liệt kê")

n= 8:    256 subsequence, sinh mất 0.000 s
n=12:   4096 subsequence, sinh mất 0.002 s
n=16:  65536 subsequence, sinh mất 0.046 s
n=100 sẽ có tới 2^100 ≈ 1.27e+30 subsequence — không thể liệt kê


## 10. Bộ test

In [14]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

s = Solution()
check("đề 1", s.findLUSlength("aba", "cdc"), 3)
check("đề 2", s.findLUSlength("aaa", "bbb"), 3)
check("đề 3", s.findLUSlength("aaa", "aaa"), -1)
check("a là subsequence của b", s.findLUSlength("ab", "abc"), 3)
check("b là subsequence của a", s.findLUSlength("abcde", "ace"), 5)
check("cùng độ dài khác nhau", s.findLUSlength("ab", "ba"), 2)
check("1 ký tự giống", s.findLUSlength("z", "z"), -1)
check("1 ký tự khác", s.findLUSlength("z", "y"), 1)
check("độ dài 100", s.findLUSlength("a" * 100, "a" * 99), 100)

PASS đề 1: 3
PASS đề 2: 3
PASS đề 3: -1
PASS a là subsequence của b: 3
PASS b là subsequence của a: 5
PASS cùng độ dài khác nhau: 2
PASS 1 ký tự giống: -1
PASS 1 ký tự khác: 1
PASS độ dài 100: 100


In [15]:
import random
random.seed(521)
count = 0
for _ in range(1500):
    a = "".join(random.choice("ab") for _ in range(random.randint(1, 7)))
    b = "".join(random.choice("ab") for _ in range(random.randint(1, 7)))
    if random.random() < 0.15:
        b = a                                          # cố ý tạo trường hợp bằng nhau
    expected = lus_bruteforce(a, b)
    assert lus_formula(a, b) == lus_check_candidates(a, b) == Solution().findLUSlength(a, b) == expected, (a, b)
    count += 1
print(f"{count} cặp ngẫu nhiên (bảng chữ 'ab', dài 1..7): công thức khớp brute force — PASS")

1500 cặp ngẫu nhiên (bảng chữ 'ab', dài 1..7): công thức khớp brute force — PASS


Bảng chữ nhỏ `"ab"` giúp tạo **nhiều** cặp có quan hệ subsequence với nhau — chính là những ca dễ làm sai nhất.

## 11. Lỗi người mới hay gặp

### Lỗi 1: nghĩ đây là bài **LCS** (Longest Common Subsequence) rồi viết DP O(n·m)
Đề hỏi uncommon, không phải common. DP vừa phức tạp vừa trả sai đáp án.

### Lỗi 2: trả `-1` khi một chuỗi là subsequence của chuỗi kia
`a = "ab"`, `b = "abc"`: `"abc"` vẫn là uncommon → đáp án 3, không phải -1.

### Lỗi 3: trả `min` thay vì `max`

### Lỗi 4: so sánh **tập ký tự** (`set(a) == set(b)`) thay vì so sánh chuỗi
`"ab"` và `"ba"` có cùng tập ký tự nhưng đáp án là 2.

In [16]:
def buggy_subseq_minus_one(a, b):
    if is_subsequence(a, b) or is_subsequence(b, a):
        return -1
    return max(len(a), len(b))

def buggy_min(a, b):
    return -1 if a == b else min(len(a), len(b))

def buggy_set(a, b):
    return -1 if set(a) == set(b) else max(len(a), len(b))

for a, b in [("ab", "abc"), ("aefd", "ab"), ("ab", "ba")]:
    print(f"{a!r:7} {b!r:6} | subseq->-1: {buggy_subseq_minus_one(a, b):2d} | min: {buggy_min(a, b):2d} "
          f"| set: {buggy_set(a, b):2d} | đúng: {Solution().findLUSlength(a, b)}")

'ab'    'abc'  | subseq->-1: -1 | min:  2 | set:  3 | đúng: 3
'aefd'  'ab'   | subseq->-1:  4 | min:  2 | set:  4 | đúng: 4
'ab'    'ba'   | subseq->-1:  2 | min:  2 | set: -1 | đúng: 2


## 12. Mẹo nhận dạng pattern

- Đề Easy nhưng định nghĩa nghe "khủng" (mọi subsequence, `2ⁿ` khả năng) → nghi ngờ có **quan sát toán học** làm bài toán sụp đổ thành vài dòng. Gợi ý "Think very simple" là tín hiệu.
- Cách tìm quan sát: viết **brute force đúng định nghĩa** trên input nhỏ, in ra bảng, tìm quy luật, rồi **chứng minh**.
- "Dài nhất thoả X" → hãy thử **ứng viên dài nhất có thể** trước (ở đây là chính các chuỗi gốc).
- Tách bạch: subsequence (giữ thứ tự, bỏ được) ≠ substring (liên tiếp) ≠ subset (không cần thứ tự).

## 13. Biến thể

### 13.1. Longest Uncommon Subsequence II (LeetCode 522)
Cho **list** chuỗi `strs`. Uncommon = subsequence của **đúng một** chuỗi trong list. Quan sát tương tự: đáp án (nếu có) là **một chuỗi nguyên vẹn trong list** — chuỗi `s` hợp lệ khi nó không là subsequence của **bất kỳ chuỗi nào khác** trong list.

In [17]:
def lus_ii(strs):
    best = -1
    for i, s in enumerate(strs):
        if all(not is_subsequence(s, t) for j, t in enumerate(strs) if j != i):
            best = max(best, len(s))
    return best

print(lus_ii(["aba", "cdc", "eae"]))    # 3
print(lus_ii(["aaa", "aaa", "aa"]))     # -1
print(lus_ii(["aabbcc", "aabbcc", "cb"]))  # 2

3
-1
2


Vì sao chỉ cần xét chuỗi nguyên vẹn? Nếu `x` là uncommon subsequence của chuỗi `s`, thì chính `s` cũng uncommon (nếu `s` là subsequence của `t` khác thì `x` cũng vậy — mâu thuẫn), và `len(s) ≥ len(x)`.

### 13.2. Is Subsequence (LeetCode 392)
Chính là hàm `is_subsequence` ở mục 2.2 — bài nền tảng nên làm cùng.

## 14. Bài tập (đáp án ở cuối)

### Bài 1 — Tính tay
Với `a = "abc"`, `b = "acb"`: `a` có là subsequence của `b` không? Đáp án của bài là bao nhiêu? Giải thích.

### Bài 2 — Điền TODO: `is_subsequence` hai con trỏ (dạng while)

In [18]:
def exercise_is_subsequence(s, t):
    i = j = 0
    # TODO 1: while i < len(s) and j < len(t)
    # TODO 2: nếu s[i] == t[j] thì i += 1; luôn j += 1
    # TODO 3: trả về i == len(s)
    pass

# assert exercise_is_subsequence("ace", "abcde") is True
# assert exercise_is_subsequence("aec", "abcde") is False

### Bài 3 — Đếm subsequence phân biệt (brute force nhỏ)
Trả về số subsequence **phân biệt, khác rỗng** của `s` (dùng `combinations`, `len(s) <= 12`).

In [19]:
def count_distinct_subsequences_small(s):
    # TODO
    pass

# assert count_distinct_subsequences_small("abc") == 7
# assert count_distinct_subsequences_small("aaa") == 3

### Bài 4 — Liệt kê **tất cả** uncommon subsequence dài nhất
Với chuỗi ngắn, trả về list (đã sắp xếp) các uncommon subsequence có độ dài lớn nhất.

In [20]:
def longest_uncommon_list(a, b):
    # TODO
    pass

# assert longest_uncommon_list("aba", "cdc") == ["aba", "cdc"]
# assert longest_uncommon_list("ab", "abc") == ["abc"]
# assert longest_uncommon_list("aa", "aa") == []

### Bài 5 — LUS II tự viết
Cài đặt lại `lus_ii(strs)` theo ý ở mục 13.1, có tối ưu: duyệt chuỗi theo độ dài giảm dần và trả ngay khi tìm được.

In [21]:
def exercise_lus_ii(strs):
    # TODO
    pass

# assert exercise_lus_ii(["aba", "cdc", "eae"]) == 3
# assert exercise_lus_ii(["aaa", "aaa", "aa"]) == -1

### Bài 6 — Mini challenge: Longest Common Subsequence (độ dài)
Để phân biệt với bài này, hãy viết DP tính độ dài LCS của `a` và `b`.

In [22]:
def lcs_length(a, b):
    # TODO: dp[i][j] = LCS của a[:i] và b[:j]
    pass

# assert lcs_length("abcde", "ace") == 3
# assert lcs_length("abc", "def") == 0

## 15. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Chạy hai con trỏ bằng tay | sau khi khớp `a`, tìm `b` trong `"cb"` rồi `c` sau đó | `a != b` thì công thức nói gì? |
| 2 | Xem mục 2.2 | điều kiện dừng có hai vế | `j += 1` mỗi vòng |
| 3 | Dùng `all_subsequences` | tập tự loại trùng | trừ 1 cho chuỗi rỗng |
| 4 | Hiệu đối xứng `SA ^ SB` | tìm độ dài lớn nhất | `sorted(x for x in U if len(x) == best)` |
| 5 | Ứng viên là chuỗi nguyên vẹn | `sorted(range(n), key=lambda i: -len(strs[i]))` | so với mọi `j != i` |
| 6 | Bảng `(len(a)+1) × (len(b)+1)` | ký tự bằng nhau → `dp[i-1][j-1] + 1` | khác nhau → `max(dp[i-1][j], dp[i][j-1])` |

## 16. Đáp án tham khảo

**Bài 1:** `"abc"` **không** là subsequence của `"acb"` (sau khi khớp `a`, `b` ở vị trí 2, không còn `c` phía sau). Vì `a != b`, đáp án = `max(3, 3) = 3`.

In [23]:
print(is_subsequence("abc", "acb"), Solution().findLUSlength("abc", "acb"), lus_bruteforce("abc", "acb"))

False 3 3


In [24]:
def answer_exercise_is_subsequence(s, t):
    i = j = 0
    while i < len(s) and j < len(t):
        if s[i] == t[j]:
            i += 1
        j += 1
    return i == len(s)

def answer_count_distinct_subsequences_small(s):
    return len(all_subsequences(s)) - 1

def answer_longest_uncommon_list(a, b):
    U = all_subsequences(a) ^ all_subsequences(b)
    if not U:
        return []
    best = max(len(x) for x in U)
    return sorted(x for x in U if len(x) == best)

def answer_exercise_lus_ii(strs):
    order = sorted(range(len(strs)), key=lambda i: -len(strs[i]))
    for i in order:
        if all(not is_subsequence(strs[i], strs[j]) for j in range(len(strs)) if j != i):
            return len(strs[i])
    return -1

def answer_lcs_length(a, b):
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[len(a)][len(b)]

assert answer_exercise_is_subsequence("ace", "abcde") is True
assert answer_exercise_is_subsequence("aec", "abcde") is False
assert answer_count_distinct_subsequences_small("abc") == 7
assert answer_count_distinct_subsequences_small("aaa") == 3
assert answer_longest_uncommon_list("aba", "cdc") == ["aba", "cdc"]
assert answer_longest_uncommon_list("ab", "abc") == ["abc"]
assert answer_longest_uncommon_list("aa", "aa") == []
assert answer_exercise_lus_ii(["aba", "cdc", "eae"]) == 3
assert answer_exercise_lus_ii(["aaa", "aaa", "aa"]) == -1
assert answer_lcs_length("abcde", "ace") == 3 and answer_lcs_length("abc", "def") == 0
print("Đáp án bài 2–6: PASS toàn bộ test!")

Đáp án bài 2–6: PASS toàn bộ test!


## 17. Template nhớ nhanh + checklist submit

```python
class Solution:
    def findLUSlength(self, a: str, b: str) -> int:
        return -1 if a == b else max(len(a), len(b))
```

Câu nhớ: **"Hai chuỗi khác nhau thì chuỗi dài hơn tự nó đã là đáp án."**

Quy trình tư duy cho "bài mẹo":
1. Viết brute force đúng định nghĩa (input nhỏ).
2. In bảng kết quả, tìm quy luật.
3. Chứng minh quy luật (giới hạn trên + đạt được).
4. Đối chiếu công thức với brute force trên nhiều input ngẫu nhiên.

### Checklist
- [ ] Phân biệt subsequence / substring.
- [ ] `a == b` → `-1`.
- [ ] Ngược lại → `max(len(a), len(b))` (không phải `min`, không phải LCS).
- [ ] Một chuỗi là subsequence của chuỗi kia **không** làm đáp án thành `-1`.
- [ ] Time `O(min(n, m))`, Space `O(1)`.